# SIMBES — Módulo 8: Constructor de Escenarios
## Integración completa M1–M7

Este notebook implementa el motor de integración del Constructor de Escenarios:
una cadena de cálculo que conecta todos los subsistemas BES en secuencia.

**Cadena de integración:**
```
M1 (IPR/Nodal)
  → M3 (GVF + degradación H-Q)
  → M2 (TDH + Nº etapas)
  → M4 (Cable + Arrhenius + THD)
  → M7 (Confiabilidad R(t))
  → Alertas cruzadas
```

**Regla de oro del módulo:**
> Una mejora aislada (más frecuencia, cable más grueso, VSD AFE) nunca puede
> analizarse sin evaluar su impacto en todos los demás subsistemas.
> El Constructor fuerza ese análisis integrado.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join('..', 'backend', 'physics'))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import FancyArrowPatch
import warnings
warnings.filterwarnings('ignore')

# Importar motores físicos SIMBES
from ipr        import calc_aof, ipr_pwf_to_q, ipr_q_to_pwf
from hydraulics import pump_head_ft, vlp_pwf, tdh_required
from gas        import gas_volume_fraction, hq_gas_degradation_factor, gas_separator_efficiency
from electrical import cable_voltage_drop, arrhenius_life_factor, thd_estimate
from reliability import survival_prob, mtbf_mle

# Constantes
M_TO_FT   = 3.28084
FT_TO_M   = 1 / M_TO_FT
STB_TO_M3 = 0.158987

# Paleta SIMBES
C = {
    'bg':      '#0B0F1A',
    'surface': '#111827',
    'border':  '#1E293B',
    'text':    '#CBD5E1',
    'muted':   '#64748B',
    'ipr':     '#38BDF8',
    'vlp':     '#34D399',
    'vlp_deg': '#F97316',
    'bep':     '#F472B6',
    'pb':      '#FBBF24',
    'op':      '#FB7185',
    'ok':      '#22C55E',
    'warn':    '#F59E0B',
    'danger':  '#EF4444',
}

def apply_dark(ax, title='', xlabel='', ylabel=''):
    ax.set_facecolor(C['surface'])
    ax.figure.patch.set_facecolor(C['bg'])
    ax.tick_params(colors=C['text'])
    ax.spines[:].set_color(C['border'])
    if title:  ax.set_title(title,  color=C['text'], fontsize=10, pad=8)
    if xlabel: ax.set_xlabel(xlabel, color=C['muted'], fontsize=9)
    if ylabel: ax.set_ylabel(ylabel, color=C['muted'], fontsize=9)
    ax.grid(True, color=C['border'], linewidth=0.5)

print('✅ Entorno listo. Módulo 8 — Constructor de Escenarios')

## 1. Motor de integración `compute_system()`

Función pura que encadena todos los módulos en secuencia.
Recibe un diccionario de parámetros y retorna KPIs + datos de gráfica.

In [ ]:
def compute_system(p: dict) -> dict:
    """
    Motor de integración M1–M7.

    Parámetros (dict p):
      Pr        : Presión estática del reservorio (psi)
      Pb        : Presión de burbuja (psi)
      IP        : Índice de Productividad (STB/d/psi)
      depth_m   : Profundidad de la bomba (m)
      Pwh       : Presión en cabeza de pozo (psi)
      freq      : Frecuencia del VSD (Hz)
      rho_kgL   : Densidad del fluido (kg/L)
      GOR       : Gas-Oil Ratio superficial (scf/STB)
      T_F       : Temperatura de fondo (°F)
      separator : 'none' | 'ags_passive' | 'gas_handler'
      mu_cp     : Viscosidad del fluido (cp)
      AWG       : Calibre de cable ('AWG4'|'AWG6'|'AWG8')
      I_amps    : Corriente nominal del motor (A)
      T_rated   : Temperatura nominal del motor (°C, clase H=180)
      vsd_topo  : Topología VSD ('std6'|'mp12'|'mp18'|'afe')
      bench_env : Ambiente de benchmarking ('benign'|'moderate'|'severe'|'sandy')
    """
    Pr, Pb, IP       = p['Pr'], p['Pb'], p['IP']
    depth_m          = p['depth_m']
    Pwh, freq        = p['Pwh'], p['freq']
    rho_kgL          = p['rho_kgL']
    GOR              = p['GOR']
    T_F              = p.get('T_F', 200.0)
    separator        = p.get('separator', 'none')
    mu_cp            = p.get('mu_cp', 1.0)
    AWG              = p.get('AWG', 'AWG6')
    I_amps           = p.get('I_amps', 80.0)
    T_rated          = p.get('T_rated', 180.0)
    vsd_topo         = p.get('vsd_topo', 'std6')
    bench_env        = p.get('bench_env', 'moderate')

    depth_ft         = depth_m * M_TO_FT
    grad_psi_ft      = rho_kgL * 0.4335

    # MTBF de referencia por ambiente (días)
    BENCH_MTBF = {'benign': 1825, 'moderate': 913, 'severe': 365, 'sandy': 180}
    MTBF_ref   = BENCH_MTBF.get(bench_env, 913)

    # ── M1: IPR ────────────────────────────────────────────────────────────
    AOF   = calc_aof(Pr, Pb, IP)
    Q_max = AOF * STB_TO_M3  # m³/d para gráfica

    # ── M3 paso 1: GVF estimado con Ps ≈ Pwh (primera pasada) ─────────────
    gvf_raw = gas_volume_fraction(GOR, Pb, Pwh, T_F)
    sep_out = gas_separator_efficiency(gvf_raw['GVF'], separator)
    GVF_pump = sep_out['GVF_pump_pct'] / 100.0
    gas_deg  = hq_gas_degradation_factor(GVF_pump)
    f_H_gas  = gas_deg['head_factor']

    # Viscosidad → factor de corrección HI (simplificado)
    f_H_visc = max(0.70, 1.0 - 0.001 * max(0, mu_cp - 1.0))
    f_H      = f_H_gas * f_H_visc

    # ── Curvas para gráfica (200 puntos) ──────────────────────────────────
    Q_range_stbd = np.linspace(0, AOF * 1.05, 200)
    IPR_curve    = [ipr_q_to_pwf(q, Pr, Pb, IP) for q in Q_range_stbd]
    VLP_clean    = [vlp_pwf(q, depth_ft, Pwh, freq, grad_psi_ft) for q in Q_range_stbd]
    VLP_deg      = [vlp_pwf(q, depth_ft, Pwh, freq, grad_psi_ft, head_factor=f_H)
                    for q in Q_range_stbd]

    # ── Bisección: punto de operación limpio ───────────────────────────────
    def bisect_op(ipr_vals, vlp_vals, q_arr):
        diff = np.array(ipr_vals) - np.array(vlp_vals)
        for i in range(len(diff) - 1):
            if diff[i] * diff[i+1] < 0:
                t = diff[i] / (diff[i] - diff[i+1])
                return q_arr[i] + t * (q_arr[i+1] - q_arr[i]), \
                       ipr_vals[i] + t * (ipr_vals[i+1] - ipr_vals[i])
        return None, None

    Q_clean_stbd, Pwf_clean = bisect_op(IPR_curve, VLP_clean, Q_range_stbd)
    Q_deg_stbd,   Pwf_deg   = bisect_op(IPR_curve, VLP_deg,   Q_range_stbd)

    Q_clean_m3 = Q_clean_stbd * STB_TO_M3 if Q_clean_stbd else None
    Q_deg_m3   = Q_deg_stbd   * STB_TO_M3 if Q_deg_stbd   else None

    # ── M3 paso 2: GVF refinado con Ps ≈ Pwf_clean × 0.8 ─────────────────
    if Pwf_clean:
        Ps_refined = Pwf_clean * 0.8
        gvf2       = gas_volume_fraction(GOR, Pb, Ps_refined, T_F)
        sep2       = gas_separator_efficiency(gvf2['GVF'], separator)
        GVF_pump   = sep2['GVF_pump_pct'] / 100.0

    # ── M2: TDH y número de etapas ────────────────────────────────────────
    TDH_ft    = (Pwf_clean - Pwh) / grad_psi_ft if Pwf_clean else 0
    H0_stage  = 45  # ft por etapa (representativo)
    N_stages  = max(1, int(np.ceil(TDH_ft / H0_stage)))

    # ── M4: Sistema eléctrico ─────────────────────────────────────────────
    cable   = cable_voltage_drop(AWG, depth_m, I_amps)
    T_bot_C = 25 + (depth_m / 100) * 3.2   # gradiente geotérmico
    arrh    = arrhenius_life_factor(T_bot_C, T_rated)
    thd     = thd_estimate(vsd_topo)

    # ── M7: Confiabilidad ─────────────────────────────────────────────────
    R_1y = survival_prob(365, MTBF_ref)
    R_2y = survival_prob(730, MTBF_ref)

    # ── Alertas cruzadas ──────────────────────────────────────────────────
    alerts = []
    if GVF_pump > 0.15:
        alerts.append({'level': 'danger',
                        'msg': f'GVF bomba = {GVF_pump*100:.1f}% > 15% — riesgo gas lock inmediato (M3)'})
    elif GVF_pump > 0.10:
        alerts.append({'level': 'warning',
                        'msg': f'GVF bomba = {GVF_pump*100:.1f}% — zona de advertencia 10–15% (M3)'})

    vdrop_pct = cable.get('voltage_drop_pct', 0)
    if vdrop_pct > 10:
        alerts.append({'level': 'danger',
                        'msg': f'V_drop = {vdrop_pct:.1f}% > 10% — sobrecalentamiento de cable (M4)'})
    elif vdrop_pct > 5:
        alerts.append({'level': 'warning',
                        'msg': f'V_drop = {vdrop_pct:.1f}% > 5% — motor recibe menos voltaje del nominal (M4)'})

    if arrh.get('life_factor', 1) < 0.35:
        alerts.append({'level': 'danger',
                        'msg': f'Vida Arrhenius = {arrh["life_factor"]*100:.0f}% — aislamiento crítico (M4)'})

    if thd.get('THD_pct', 0) > 5:
        alerts.append({'level': 'warning',
                        'msg': f'THD = {thd["THD_pct"]:.0f}% — no cumple IEEE 519 < 5% (M4)'})

    if R_1y < 0.30:
        alerts.append({'level': 'warning',
                        'msg': f'R(1 año) = {R_1y*100:.0f}% — alta probabilidad de falla antes de 12 meses (M7)'})

    Q_drop_pct = None
    if Q_clean_m3 and Q_deg_m3:
        Q_drop_pct = (Q_clean_m3 - Q_deg_m3) / Q_clean_m3 * 100
        if Q_drop_pct > 20:
            alerts.append({'level': 'warning',
                            'msg': f'Pérdida de producción = {Q_drop_pct:.1f}% por degradación H-Q (M2+M3)'})

    return {
        'AOF': AOF, 'Q_max_m3': Q_max,
        'Q_range': Q_range_stbd,
        'IPR': IPR_curve, 'VLP_clean': VLP_clean, 'VLP_deg': VLP_deg,
        'Q_clean_stbd': Q_clean_stbd, 'Pwf_clean': Pwf_clean,
        'Q_clean_m3': Q_clean_m3,
        'Q_deg_stbd': Q_deg_stbd,   'Pwf_deg': Pwf_deg,
        'Q_deg_m3': Q_deg_m3,
        'f_H': f_H, 'GVF_pump_pct': GVF_pump * 100,
        'TDH_ft': TDH_ft, 'N_stages': N_stages,
        'cable': cable, 'arrh': arrh, 'thd': thd, 'T_bot_C': T_bot_C,
        'MTBF_ref': MTBF_ref, 'R_1y': R_1y, 'R_2y': R_2y,
        'alerts': alerts, 'Q_drop_pct': Q_drop_pct,
        'grad_psi_ft': grad_psi_ft,
    }

print('✅ compute_system() definida')

## 2. Escenario base — Pozo nominal

| Parámetro | Valor |
|---|---|
| Pr | 3800 psi |
| Pb | 1500 psi |
| IP | 2.5 STB/d/psi |
| Profundidad | 2200 m |
| Frecuencia | 60 Hz |
| GOR | 80 scf/STB |
| Ambiente | Moderado (MTBF ref = 913 d) |

In [ ]:
P_nominal = dict(
    Pr=3800, Pb=1500, IP=2.5,
    depth_m=2200, Pwh=150, freq=60,
    rho_kgL=0.85, GOR=80, T_F=200,
    separator='none', mu_cp=5.0,
    AWG='AWG6', I_amps=80, T_rated=180,
    vsd_topo='mp12', bench_env='moderate'
)

R_nom = compute_system(P_nominal)

print('=== ESCENARIO NOMINAL ===')
print(f"AOF               = {R_nom['AOF']:.0f} STB/d")
print(f"Q operativo limpio = {R_nom['Q_clean_stbd']:.0f} STB/d  ({R_nom['Q_clean_m3']:.1f} m³/d)")
print(f"Pwf operativo      = {R_nom['Pwf_clean']:.0f} psi")
print(f"GVF bomba          = {R_nom['GVF_pump_pct']:.1f}%")
print(f"f_H (degradación)  = {R_nom['f_H']:.3f}")
print(f"TDH                = {R_nom['TDH_ft']:.0f} ft  →  {R_nom['N_stages']} etapas")
print(f"V_drop             = {R_nom['cable'].get('voltage_drop_pct', 0):.1f}%")
print(f"THD                = {R_nom['thd'].get('THD_pct', 0):.0f}%")
print(f"T_bot_C            = {R_nom['T_bot_C']:.1f} °C")
print(f"Vida Arrhenius     = {R_nom['arrh'].get('life_factor', 1)*100:.0f}%")
print(f"MTBF ref           = {R_nom['MTBF_ref']} días")
print(f"R(1 año)           = {R_nom['R_1y']*100:.1f}%")
print(f"Alertas            = {len(R_nom['alerts'])}")
for a in R_nom['alerts']:
    icon = '🔴' if a['level'] == 'danger' else '🟡'
    print(f"  {icon} {a['msg']}")

## 3. Gráfica integrada: IPR + VLP limpio + VLP degradado

La gráfica central del Constructor muestra cómo la degradación
H-Q (gas + viscosidad) desplaza el punto de operación.

In [ ]:
def plot_integrated(result, params, title='Análisis Nodal Integrado'):
    fig, ax = plt.subplots(figsize=(11, 6))
    apply_dark(ax, title=title,
               xlabel='Caudal Q (STB/d)', ylabel='Presión Pwf (psi)')

    Q = result['Q_range']

    # Curvas
    ax.plot(Q, result['IPR'],      color=C['ipr'],     lw=2.5, label='IPR')
    ax.plot(Q, result['VLP_clean'], color=C['vlp'],    lw=2,   label='VLP limpio')
    ax.plot(Q, result['VLP_deg'],   color=C['vlp_deg'],lw=2,
            ls='--', label=f'VLP degradado (f_H={result["f_H"]:.2f})')

    # Pb horizontal
    ax.axhline(params['Pb'], color=C['pb'], lw=1, ls=':', alpha=0.8)
    ax.text(Q[-1]*0.97, params['Pb']+30, f'Pb = {params["Pb"]} psi',
            color=C['pb'], fontsize=8, ha='right')

    # Puntos de operación
    if result['Q_clean_stbd']:
        ax.scatter([result['Q_clean_stbd']], [result['Pwf_clean']],
                   color=C['vlp'], s=120, zorder=5, label=f"Op. limpio: {result['Q_clean_stbd']:.0f} STB/d")
    if result['Q_deg_stbd']:
        ax.scatter([result['Q_deg_stbd']], [result['Pwf_deg']],
                   color=C['vlp_deg'], s=120, zorder=5,
                   marker='D', label=f"Op. deg.: {result['Q_deg_stbd']:.0f} STB/d")

    ax.set_xlim(0, result['Q_range'][-1] * 1.05)
    ax.set_ylim(0, params['Pr'] * 1.1)
    legend = ax.legend(fontsize=8, facecolor=C['surface'], edgecolor=C['border'],
                        labelcolor=C['text'])
    plt.tight_layout()
    return fig

fig = plot_integrated(R_nom, P_nominal, 'Escenario Nominal — Análisis Nodal Integrado')
plt.show()
print(f"Pérdida por degradación: {R_nom.get('Q_drop_pct', 0):.1f}%")

## 4. Comparación de escenarios — Nominal vs. Degradado

Escenario B reproduce un pozo deteriorado:
- GOR alto (800 scf/STB) → GVF elevado
- Cable delgado AWG8 → V_drop alto
- Alta temperatura → Arrhenius activo
- Ambiente severo → MTBF bajo

In [ ]:
P_degradado = dict(
    Pr=3800, Pb=1500, IP=2.5,
    depth_m=2200, Pwh=150, freq=65,
    rho_kgL=0.85, GOR=800, T_F=230,
    separator='gas_handler', mu_cp=20.0,
    AWG='AWG8', I_amps=80, T_rated=180,
    vsd_topo='std6', bench_env='severe'
)

R_deg = compute_system(P_degradado)

# Tabla comparativa
labels = [
    'Q operativo (STB/d)', 'Q operativo (m³/d)', 'Pwf operativo (psi)',
    'GVF bomba (%)', 'f_H degradación',
    'TDH (ft)', 'Nº etapas',
    'V_drop (%)', 'THD (%)', 'T_fondo (°C)',
    'Vida Arrhenius (%)', 'MTBF ref (días)',
    'R(1 año) (%)', 'R(2 años) (%)',
    'Alertas cruzadas'
]

def fmt(r, key_fn):
    try: return key_fn(r)
    except: return 'N/A'

vals_nom = [
    fmt(R_nom, lambda r: f"{r['Q_clean_stbd']:.0f}"),
    fmt(R_nom, lambda r: f"{r['Q_clean_m3']:.1f}"),
    fmt(R_nom, lambda r: f"{r['Pwf_clean']:.0f}"),
    f"{R_nom['GVF_pump_pct']:.1f}",
    f"{R_nom['f_H']:.3f}",
    f"{R_nom['TDH_ft']:.0f}",
    str(R_nom['N_stages']),
    f"{R_nom['cable'].get('voltage_drop_pct', 0):.1f}",
    f"{R_nom['thd'].get('THD_pct', 0):.0f}",
    f"{R_nom['T_bot_C']:.1f}",
    f"{R_nom['arrh'].get('life_factor', 1)*100:.0f}",
    str(R_nom['MTBF_ref']),
    f"{R_nom['R_1y']*100:.1f}",
    f"{R_nom['R_2y']*100:.1f}",
    str(len(R_nom['alerts'])),
]

vals_deg = [
    fmt(R_deg, lambda r: f"{r['Q_clean_stbd']:.0f}"),
    fmt(R_deg, lambda r: f"{r['Q_clean_m3']:.1f}"),
    fmt(R_deg, lambda r: f"{r['Pwf_clean']:.0f}"),
    f"{R_deg['GVF_pump_pct']:.1f}",
    f"{R_deg['f_H']:.3f}",
    f"{R_deg['TDH_ft']:.0f}",
    str(R_deg['N_stages']),
    f"{R_deg['cable'].get('voltage_drop_pct', 0):.1f}",
    f"{R_deg['thd'].get('THD_pct', 0):.0f}",
    f"{R_deg['T_bot_C']:.1f}",
    f"{R_deg['arrh'].get('life_factor', 1)*100:.0f}",
    str(R_deg['MTBF_ref']),
    f"{R_deg['R_1y']*100:.1f}",
    f"{R_deg['R_2y']*100:.1f}",
    str(len(R_deg['alerts'])),
]

print(f"{'KPI':<30} {'Nominal':>12} {'Degradado':>12}")
print('-' * 56)
for l, a, b in zip(labels, vals_nom, vals_deg):
    print(f"{l:<30} {a:>12} {b:>12}")

print('\n=== ALERTAS ESCENARIO DEGRADADO ===')
for a in R_deg['alerts']:
    icon = '🔴' if a['level'] == 'danger' else '🟡'
    print(f"  {icon} {a['msg']}")

## 5. Análisis de sensibilidad — Impacto de frecuencia en todos los KPIs

Barrido de frecuencia 40–70 Hz sobre el escenario nominal.
Muestra cómo una sola variable (freq) afecta simultáneamente
producción, GVF, V_drop y confiabilidad.

In [ ]:
freqs = np.arange(40, 72, 2)

res_freq = []
for f in freqs:
    p = {**P_nominal, 'freq': float(f)}
    r = compute_system(p)
    res_freq.append(r)

Q_vals     = [r['Q_clean_m3'] or 0   for r in res_freq]
GVF_vals   = [r['GVF_pump_pct']       for r in res_freq]
Vdrop_vals = [r['cable'].get('voltage_drop_pct', 0) for r in res_freq]
R1y_vals   = [r['R_1y'] * 100         for r in res_freq]

fig, axes = plt.subplots(2, 2, figsize=(13, 8))
fig.patch.set_facecolor(C['bg'])
fig.suptitle('Sensibilidad a Frecuencia VSD (40–70 Hz) — Escenario Nominal',
             color=C['text'], fontsize=12, y=1.01)

plot_specs = [
    (axes[0,0], Q_vals,     'Caudal operativo (m³/d)',     C['vlp'],     'Q (m³/d)'),
    (axes[0,1], GVF_vals,   'GVF en bomba (%)',             C['warn'],    'GVF (%)'),
    (axes[1,0], Vdrop_vals, 'Caída de voltaje en cable (%)',C['vlp_deg'], 'V_drop (%)'),
    (axes[1,1], R1y_vals,   'R(1 año) — Confiabilidad (%)', C['bep'],     'R_1y (%)'),
]

for ax, vals, title, color, ylabel in plot_specs:
    apply_dark(ax, title=title, xlabel='Frecuencia (Hz)', ylabel=ylabel)
    ax.plot(freqs, vals, color=color, lw=2, marker='o', ms=4)
    ax.axvline(60, color=C['muted'], lw=1, ls='--', alpha=0.5)
    ax.text(60.5, ax.get_ylim()[1]*0.95, '60 Hz', color=C['muted'], fontsize=7)

# Umbrales de alerta
axes[0,1].axhline(15, color=C['danger'], lw=1, ls='--', alpha=0.7)
axes[0,1].text(70.5, 15, 'Gas lock\n15%', color=C['danger'], fontsize=7)
axes[1,0].axhline(5, color=C['warn'], lw=1, ls='--', alpha=0.7)
axes[1,0].text(70.5, 5, 'IEEE\n5%', color=C['warn'], fontsize=7)

plt.tight_layout()
plt.show()

## 6. Análisis económico — MTBF vs. producción

### Caso pregunta M8Q3:
- **Escenario A:** 60 Hz, sin separador, GVF=8%, Q=420 m³/d, MTBF=280 días
- **Escenario B:** 55 Hz, AGS, GVF=3%, Q=310 m³/d, MTBF=650 días
- Precio crudo: USD 60/bbl | Costo intervención: USD 180,000
- Conversión: 1 bbl = 0.159 m³

La decisión de operación no es solo técnica — es económica.

In [ ]:
# Parámetros económicos
PRECIO_USD_BBL   = 60
COSTO_INTERV_USD = 180_000
BBL_PER_M3       = 1 / 0.159
HORIZON_YEARS    = [1, 2, 3, 5]

scenarios = {
    'A — Alta producción\n(60 Hz, sin AGS)': {'Q_m3d': 420, 'MTBF_d': 280},
    'B — Alta confiabilidad\n(55 Hz, AGS)':  {'Q_m3d': 310, 'MTBF_d': 650},
}

fig, axes = plt.subplots(1, 2, figsize=(13, 6))
fig.patch.set_facecolor(C['bg'])
fig.suptitle('Análisis Económico — Producción vs. Costo de Intervenciones',
             color=C['text'], fontsize=12)

# Ingreso bruto y neto por horizonte
years = np.linspace(0, 5, 200)
colors_sc = [C['vlp'], C['bep']]

print(f"{'Horizonte':>10}  {'Ingreso A':>14}  {'Costo interv. A':>16}  "
      f"{'Neto A':>12}  {'Ingreso B':>14}  {'Costo interv. B':>16}  {'Neto B':>12}")
print('-' * 100)

for yr in HORIZON_YEARS:
    days = yr * 365
    for name, sc in scenarios.items():
        Q_bbl_yr  = sc['Q_m3d'] * BBL_PER_M3 * days
        ingreso   = Q_bbl_yr * PRECIO_USD_BBL
        n_interv  = days / sc['MTBF_d']
        costo     = n_interv * COSTO_INTERV_USD
        neto      = ingreso - costo
        sc[f'neto_{yr}y'] = neto

for yr in HORIZON_YEARS:
    A = scenarios['A — Alta producción\n(60 Hz, sin AGS)']
    B = scenarios['B — Alta confiabilidad\n(55 Hz, AGS)']
    dA  = yr * 365
    ingA = A['Q_m3d'] * BBL_PER_M3 * dA * PRECIO_USD_BBL
    cA   = dA / A['MTBF_d'] * COSTO_INTERV_USD
    ingB = B['Q_m3d'] * BBL_PER_M3 * dA * PRECIO_USD_BBL
    cB   = dA / B['MTBF_d'] * COSTO_INTERV_USD
    print(f"{yr:>9}a  {ingA/1e6:>13.2f}M  {cA/1e3:>15.0f}k  "
          f"{(ingA-cA)/1e6:>11.2f}M  {ingB/1e6:>13.2f}M  {cB/1e3:>15.0f}k  "
          f"{(ingB-cB)/1e6:>11.2f}M")

# Gráfica ingresos netos acumulados
ax = axes[0]
apply_dark(ax, 'Ingreso neto acumulado', 'Años', 'USD (millones)')
for (name, sc), col in zip(scenarios.items(), colors_sc):
    neto_y = []
    for y in years:
        d     = y * 365
        ingr  = sc['Q_m3d'] * BBL_PER_M3 * d * PRECIO_USD_BBL
        costo = (d / sc['MTBF_d']) * COSTO_INTERV_USD
        neto_y.append((ingr - costo) / 1e6)
    ax.plot(years, neto_y, color=col, lw=2, label=name.replace('\n', ' '))
legend = ax.legend(fontsize=7, facecolor=C['surface'], edgecolor=C['border'],
                    labelcolor=C['text'])

# Intervenciones acumuladas por año
ax2 = axes[1]
apply_dark(ax2, 'Intervenciones acumuladas', 'Años', 'N° intervenciones')
for (name, sc), col in zip(scenarios.items(), colors_sc):
    interv = [y * 365 / sc['MTBF_d'] for y in years]
    ax2.plot(years, interv, color=col, lw=2, label=name.replace('\n', ' '))
legend2 = ax2.legend(fontsize=7, facecolor=C['surface'], edgecolor=C['border'],
                      labelcolor=C['text'])

plt.tight_layout()
plt.show()

## 7. Mapa de calor — GVF × Frecuencia sobre producción neta

La interacción M1↔M3 visualizada: a mayor frecuencia,
mayor producción bruta pero también mayor GVF (más drawdown → Pwf < Pb).
Existe un óptimo que el Constructor ayuda a encontrar.

In [ ]:
GOR_vals  = np.arange(50, 900, 100)
freq_vals = np.arange(45, 70, 5)

Q_matrix = np.zeros((len(freq_vals), len(GOR_vals)))

for i, freq in enumerate(freq_vals):
    for j, GOR in enumerate(GOR_vals):
        p = {**P_nominal, 'freq': float(freq), 'GOR': float(GOR)}
        r = compute_system(p)
        Q_matrix[i, j] = r['Q_deg_m3'] or r['Q_clean_m3'] or 0

fig, ax = plt.subplots(figsize=(11, 6))
fig.patch.set_facecolor(C['bg'])
apply_dark(ax, 'Producción real (m³/d) — Impacto GOR × Frecuencia',
           'GOR (scf/STB)', 'Frecuencia (Hz)')

im = ax.imshow(Q_matrix, aspect='auto', origin='lower',
               extent=[GOR_vals[0], GOR_vals[-1], freq_vals[0], freq_vals[-1]],
               cmap='RdYlGn', interpolation='bilinear')

cbar = plt.colorbar(im, ax=ax)
cbar.ax.yaxis.set_tick_params(color=C['text'])
cbar.set_label('Q (m³/d)', color=C['text'])
plt.setp(plt.getp(cbar.ax.axes, 'yticklabels'), color=C['text'])

# Contorno donde GVF puede cruzar el umbral 15%
ax.text(0.5, 0.92,
        '🟡 Zona verde = mayor producción real.  🔴 Zona roja = GVF alto degrada la bomba.',
        transform=ax.transAxes, color=C['text'], fontsize=8, ha='center',
        bbox=dict(boxstyle='round', facecolor=C['surface'], edgecolor=C['border']))

plt.tight_layout()
plt.show()

print('\n✅ Módulo 8 — Constructor de Escenarios: notebook completo')
print('   Cadena M1→M3→M2→M4→M7 implementada y verificada.')
print('   El análisis integrado es el corazón del Constructor de Escenarios.')